In [4]:
!pip install transformers datasets accelerate evaluate

In [5]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from datasets import Dataset, DatasetDict
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import evaluate

In [6]:
# ==========================================
# 1. LOAD AND COMBINE DATASETS
# ==========================================
print("Loading and preparing datasets...")

# 1A. Load Global Dataset (WELFake)
# Labels: 0 = Fake, 1 = Real
df_global = pd.read_csv('WELFake_Dataset.csv').dropna(subset=['title', 'text'])
df_global['text'] = df_global['title'].astype(str) + " " + df_global['text'].astype(str)
df_global = df_global[['text', 'label']]
df_global['label'] = df_global['label'].astype(int)

# 1B. Load Indian Dataset (IFND)
df_indian = pd.read_csv('news_dataset.csv').dropna(subset=['text', 'label'])
df_indian = df_indian[['text', 'label']]

# CRITICAL FIX: Map IFND string labels to match WELFake integers exactly
# 'FAKE' -> 0, 'REAL' -> 1
df_indian['label'] = df_indian['label'].map({'REAL': 1, 'FAKE': 0})
df_indian = df_indian.dropna(subset=['label']) # Drop any parsing errors
df_indian['label'] = df_indian['label'].astype(int)

# 1C. Combine and Oversample
df_indian_weighted = pd.concat([df_indian] * 5, ignore_index=True)
df_combined = pd.concat([df_global, df_indian_weighted], ignore_index=True)
df_combined = shuffle(df_combined, random_state=42).reset_index(drop=True)

# 1D. Train/Validation Split
train_df, val_df = train_test_split(df_combined, test_size=0.2, random_state=42)
print(f"Total training articles: {len(train_df)} | Total validation articles: {len(val_df)}")

Loading and preparing datasets...
Total training articles: 72113 | Total validation articles: 18029


In [7]:
# ==========================================
# 2. HUGGING FACE DATASET CONVERSION
# ==========================================
print("Converting to Hugging Face Dataset format...")
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

hf_datasets = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})


Converting to Hugging Face Dataset format...


In [8]:
# ==========================================
# 3. TOKENIZATION (DISTILBERT)
# ==========================================
print("Tokenizing texts...")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    # Truncate to 512 tokens (DistilBERT's maximum context window)
    return tokenizer(examples['text'], truncation=True, max_length=512)

tokenized_datasets = hf_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Remove pandas string columns to save RAM during PyTorch tensor operations
tokenized_datasets = tokenized_datasets.remove_columns(["text", "__index_level_0__"])
tokenized_datasets.set_format("torch")

Tokenizing texts...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/72113 [00:00<?, ? examples/s]

Map:   0%|          | 0/18029 [00:00<?, ? examples/s]

In [9]:
# ==========================================
# 4. MODEL INITIALIZATION & METRICS
# ==========================================
print("Initializing DistilBERT...")
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Initializing DistilBERT...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# ==========================================
# 5. TRAINING ARGUMENTS & TRAINER SETUP
# ==========================================
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True, # Mixed precision memory saver
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [13]:
# ==========================================
# 6. EXECUTE TRAINING & SAVE
# ==========================================
print("Starting training loop...")
trainer.train()

print("Saving final model artifacts...")
model.save_pretrained("./distilbert_fakenews_final")
tokenizer.save_pretrained("./distilbert_fakenews_final")
print("Pipeline Complete! Download the './distilbert_fakenews_final' folder.")

Starting training loop...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.021598,0.015770,0.995729
2,0.019273,0.016534,0.995507
3,0.008834,0.016309,0.997171


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving final model artifacts...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Pipeline Complete! Download the './distilbert_fakenews_final' folder.
